# Rice gene IDs, genomic sequences and transcript sequences in Google Colab

This notebook prepares reference sequence data for a later machine-learning study. It does not train a lodging classifier or measure stem expression.

**Inputs:** upload your `osa1_r7.all_models.gff3` (or its gzip version) in Step 2. Step 3 downloads the matching complete genome `osa1_r7.asm.fa.gz` directly from the official provider, so a genome upload is unnecessary. Use a CPU Colab runtime; a GPU is unnecessary for extraction.

Your inspected annotation contains **55,986 genes and 66,338 mRNA models**, with no embedded FASTA. MOC1 is `LOC_Os06g40780` in this annotation. Sequences are read from the genome, not inferred by ML.

Outputs distinguish:
- **Genomic gene sequence:** annotated start-to-end region, including introns and annotated UTRs, oriented 5′→3′ on the gene strand. It does not include unannotated promoter or flanking regions.
- **Transcript/cDNA:** annotated exons joined together and oriented 5′→3′; includes annotated UTRs and excludes introns. Every annotated isoform is retained.

These are reference-model sequences, not proof of experimentally complete transcripts or stem-specific expression. For CDS/protein sequences, the provider supplies matching `all_models.cds.fa.gz` and `all_models.pep.fa.gz`; cDNA must not be translated as if it were entirely coding.

Run cells in order. Download the output ZIP or copy it to Drive before the Colab session expires.


In [ ]:
from pathlib import Path
import os, gzip, csv, json, hashlib, shutil, itertools
from collections import Counter, defaultdict
from urllib.parse import unquote

ROOT = Path(os.environ.get("RICE_PROJECT_DIR", "/content/rice_gene_project"))
INPUT = ROOT / "input"
OUTPUT = ROOT / "output"
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT)
print("Free disk GB:", round(shutil.disk_usage(ROOT).free / 1e9, 2))


## 2. Upload the GFF3
Choose only `osa1_r7.all_models.gff3` or `osa1_r7.all_models.gff3.gz`. Uploading an annotation alone cannot supply DNA bases.

In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one GFF3 file.")
name, data = next(iter(uploaded.items()))
gff_path = INPUT / Path(name).name
if not gff_path.name.lower().endswith((".gff3", ".gff3.gz", ".gff", ".gff.gz")):
    raise ValueError("Expected a GFF3 annotation file.")
gff_path.write_bytes(data)
del uploaded, data
print("Annotation:", gff_path)


## 3. Download the matching complete genome FASTA
This cell downloads `osa1_r7.asm.fa.gz` directly from the official Rice Genome Annotation Project and explicitly resets `genome_path`. No genome upload is needed. It checks the first chromosome header before replacing the downloaded file. Step 4 checks the full DNA alphabet, chromosome names and coordinate bounds. Run this cell before Step 4; downloading needs internet access.


In [ ]:
from urllib.request import Request, urlopen
import gzip, shutil

url = "https://rice.uga.edu/osa1r7_download/osa1_r7.asm.fa.gz"
genome_path = INPUT / "osa1_r7.asm.fa.gz"
temporary_path = INPUT / "osa1_r7.asm.fa.gz.part"

print("Downloading the complete rice genome...")
request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urlopen(request, timeout=120) as response, open(temporary_path, "wb") as target:
    shutil.copyfileobj(response, target)

with gzip.open(temporary_path, "rt") as handle:
    header = handle.readline().strip()
expected_ids = {f"Chr{i}" for i in range(1, 13)} | {"ChrUn", "ChrSy"}
if not header.startswith(">") or header[1:].split()[0] not in expected_ids:
    raise ValueError(f"Expected chromosome FASTA, received: {header[:120]}")

temporary_path.replace(genome_path)
print("Genome file:", genome_path)
print("First header:", header)
print("Now run Step 4, followed by Steps 5–7.")


## 4. Parse the annotation and genome
Coordinates in GFF3 are **1-based and inclusive**. The code checks IDs, parent relationships, chromosome names and coordinate bounds. Minus-strand sequences are reverse-complemented. A missing chromosome or malformed hierarchy stops extraction instead of silently dropping genes.

In [ ]:
def open_text(path):
    return gzip.open(path, "rt") if str(path).endswith(".gz") else open(path, "rt")

def parse_gff(path):
    genes, transcripts = {}, {}
    exons = defaultdict(list)
    feature_counts = Counter()
    with open_text(path) as handle:
        for lineno, line in enumerate(handle, 1):
            if line.startswith("##FASTA"):
                break
            if not line.strip() or line.startswith("#"):
                continue
            cols = line.rstrip("\r\n").split("\t")
            if len(cols) != 9:
                raise ValueError(f"GFF3 line {lineno}: expected 9 columns.")
            chrom, source, kind, start, end, score, strand, phase, attrs_raw = cols
            attrs = {k: unquote(v) for k, v in
                     (item.split("=", 1) for item in attrs_raw.split(";") if "=" in item)}
            feature_counts[kind] += 1
            rec = {"chromosome": chrom, "start": int(start), "end": int(end),
                   "strand": strand, "description": attrs.get("Note", "")}
            if rec["start"] < 1 or rec["end"] < rec["start"]:
                raise ValueError(f"Invalid interval on GFF3 line {lineno}.")
            if kind in ("gene", "mRNA", "transcript"):
                ident = attrs.get("ID")
                if not ident:
                    raise ValueError(f"Missing ID on line {lineno}.")
                target = genes if kind == "gene" else transcripts
                if ident in target:
                    raise ValueError(f"Duplicate feature ID: {ident}")
                if kind != "gene":
                    parents = attrs.get("Parent", "").split(",")
                    if len(parents) != 1 or not parents[0]:
                        raise ValueError(f"Expected one parent gene for {ident}.")
                    rec["gene_id"] = parents[0]
                target[ident] = rec
            elif kind == "exon":
                parents = attrs.get("Parent", "").split(",")
                if not parents[0]:
                    raise ValueError(f"Exon without Parent on line {lineno}.")
                for parent in parents:
                    exons[parent].append((chrom, int(start), int(end), strand))
    for tid, rec in transcripts.items():
        if rec["gene_id"] not in genes:
            raise ValueError(f"Unknown parent gene for {tid}.")
        gene = genes[rec["gene_id"]]
        if rec["chromosome"] != gene["chromosome"] or rec["strand"] != gene["strand"]:
            raise ValueError(f"Transcript/gene chromosome or strand mismatch: {tid}")
        if not gene["start"] <= rec["start"] <= rec["end"] <= gene["end"]:
            raise ValueError(f"Transcript outside gene interval: {tid}")
        if not exons[tid]:
            raise ValueError(f"No exons annotated for {tid}.")
    orphan_exons = set(exons) - set(transcripts)
    if orphan_exons:
        raise ValueError(f"Exons reference unknown transcripts: {sorted(orphan_exons)[:5]}")
    return genes, transcripts, exons, feature_counts

def load_genome(path):
    genome, name, parts = {}, None, []
    with open_text(path) as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if name is not None:
                    genome[name] = "".join(parts).upper()
                name = line[1:].split()[0]
                if name in genome:
                    raise ValueError(f"Duplicate FASTA ID: {name}")
                parts = []
            else:
                if name is None:
                    raise ValueError("FASTA sequence before its header.")
                parts.append(line)
        if name is not None:
            genome[name] = "".join(parts).upper()
    if not genome:
        raise ValueError("No FASTA records found.")
    allowed = set("ACGTRYSWKMBDHVN")
    for name, sequence in genome.items():
        if set(sequence) - allowed:
            raise ValueError(f"Unexpected alphabet in {name}; verify this is DNA FASTA.")
    return genome

COMPLEMENT = str.maketrans("ACGTRYSWKMBDHVN", "TGCAYRSWMKVHDBN")
def reverse_complement(sequence):
    return sequence.translate(COMPLEMENT)[::-1]

def extract_interval(genome, chrom, start, end, strand):
    if chrom not in genome:
        raise ValueError(f"Chromosome {chrom} is absent from genome FASTA.")
    if start < 1 or end < start or end > len(genome[chrom]):
        raise ValueError(f"Out-of-bounds interval: {chrom}:{start}-{end}")
    if strand not in ("+", "-"):
        raise ValueError(f"Unsupported strand: {strand}")
    sequence = genome[chrom][start - 1:end]
    return reverse_complement(sequence) if strand == "-" else sequence

def transcript_sequence(tid, transcripts, exons, genome):
    rec = transcripts[tid]
    blocks = sorted(set(exons[tid]), key=lambda x: (x[1], x[2]))
    previous_end = 0
    for chrom, start, end, strand in blocks:
        if chrom != rec["chromosome"] or strand != rec["strand"]:
            raise ValueError(f"Exon chromosome/strand mismatch: {tid}")
        if not rec["start"] <= start <= end <= rec["end"]:
            raise ValueError(f"Exon outside transcript interval: {tid}")
        if start <= previous_end:
            raise ValueError(f"Overlapping exons: {tid}")
        previous_end = end
    sequence = "".join(extract_interval(genome, chrom, start, end, "+")
                       for chrom, start, end, strand in blocks)
    return reverse_complement(sequence) if rec["strand"] == "-" else sequence

genes, transcripts, exons, feature_counts = parse_gff(gff_path)
genome = load_genome(genome_path)
missing = {r["chromosome"] for r in genes.values()} - set(genome)
if missing:
    raise ValueError(f"Missing FASTA chromosomes: {sorted(missing)}. Use matching full assembly.")
for gid, r in genes.items():
    extract_interval(genome, r["chromosome"], r["start"], r["end"], r["strand"])
print("Genes:", len(genes), "Transcripts:", len(transcripts))
print("Feature counts:", dict(feature_counts))
print("Genome records:", list(genome))


## 5. Export every gene and transcript, and prepare numerical features

`gene_sequences.csv.gz` contains **one row per gene with its full annotated genomic sequence**. `transcript_sequences.csv.gz` contains **one row per transcript with gene ID and cDNA sequence**, so isoforms are not lost.

The feature table uses one representative transcript per gene: **longest spliced cDNA**, with transcript ID breaking length ties. This choice is explicitly different from choosing longest CDS and is not evidence of the most expressed isoform. Sequence length, GC fraction, ambiguous-base fraction, exon count and normalized 3-mer frequencies are features only; there is no lodging label in this annotation.

Genomic and transcript FASTA files are also exported. Numeric features use cDNA sequences consistently; genomic sequences remain separately identified.

In [ ]:
KMERS = ["".join(p) for p in itertools.product("ACGT", repeat=3)]
def basic_stats(sequence):
    counts = Counter(sequence)
    valid = sum(counts[b] for b in "ACGT")
    return {"length_bp": len(sequence),
            "gc_fraction_acgt": ((counts["G"] + counts["C"]) / valid) if valid else "",
            "ambiguous_fraction": (len(sequence) - valid) / len(sequence) if sequence else ""}

def kmer_frequencies(sequence):
    counts = Counter(sequence[i:i + 3] for i in range(len(sequence) - 2)
                     if all(b in "ACGT" for b in sequence[i:i + 3]))
    denominator = sum(counts.values())
    return {f"kmer_{k}": counts[k] / denominator if denominator else 0.0 for k in KMERS}

def write_fasta(handle, ident, sequence, description=""):
    handle.write(f">{ident} {description}\n")
    for i in range(0, len(sequence), 60):
        handle.write(sequence[i:i + 60] + "\n")

def export_all(genes, transcripts, exons, genome, output):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    gene_fields = ["gene_id", "chromosome", "start", "end", "strand", "description",
                   "length_bp", "gc_fraction_acgt", "ambiguous_fraction", "sequence"]
    with gzip.open(output / "gene_genomic.fasta.gz", "wt", compresslevel=1) as fasta, \
         gzip.open(output / "gene_sequences.csv.gz", "wt", newline="", compresslevel=1) as table, \
         open(output / "gene_ids.csv", "w", newline="") as ids:
        writer = csv.DictWriter(table, fieldnames=gene_fields)
        writer.writeheader()
        id_writer = csv.writer(ids); id_writer.writerow(["gene_id"])
        for gid, rec in genes.items():
            sequence = extract_interval(genome, rec["chromosome"], rec["start"], rec["end"], rec["strand"])
            writer.writerow({"gene_id": gid, **rec, **basic_stats(sequence), "sequence": sequence})
            id_writer.writerow([gid])
            write_fasta(fasta, gid, sequence,
                        f'genomic {rec["chromosome"]}:{rec["start"]}-{rec["end"]} strand={rec["strand"]}')
    representatives = {}
    tx_count = Counter(r["gene_id"] for r in transcripts.values())
    tx_fields = ["gene_id", "transcript_id", "chromosome", "strand", "exon_count",
                 "length_bp", "gc_fraction_acgt", "ambiguous_fraction", "sequence"]
    with gzip.open(output / "transcript_cdna.fasta.gz", "wt", compresslevel=1) as fasta, \
         gzip.open(output / "transcript_sequences.csv.gz", "wt", newline="", compresslevel=1) as table:
        writer = csv.DictWriter(table, fieldnames=tx_fields); writer.writeheader()
        for tid, rec in transcripts.items():
            sequence = transcript_sequence(tid, transcripts, exons, genome)
            row = {"gene_id": rec["gene_id"], "transcript_id": tid,
                   "chromosome": rec["chromosome"], "strand": rec["strand"],
                   "exon_count": len(set(exons[tid])), **basic_stats(sequence), "sequence": sequence}
            writer.writerow(row)
            write_fasta(fasta, tid, sequence, f'gene_id={rec["gene_id"]} spliced_cdna')
            previous = representatives.get(rec["gene_id"])
            if previous is None or len(sequence) > len(previous["sequence"]) or \
               (len(sequence) == len(previous["sequence"]) and tid < previous["transcript_id"]):
                representatives[rec["gene_id"]] = row
    feat_fields = ["gene_id", "representative_transcript_id", "genomic_length_bp",
                   "transcript_length_bp", "gc_fraction_acgt", "ambiguous_fraction",
                   "exon_count", "intron_count", "transcript_count"] + [f"kmer_{k}" for k in KMERS]
    with open(output / "ml_gene_features.csv", "w", newline="") as table:
        writer = csv.DictWriter(table, fieldnames=feat_fields); writer.writeheader()
        for gid, row in representatives.items():
            writer.writerow({"gene_id": gid, "representative_transcript_id": row["transcript_id"],
                "genomic_length_bp": genes[gid]["end"] - genes[gid]["start"] + 1,
                "transcript_length_bp": row["length_bp"],
                "gc_fraction_acgt": row["gc_fraction_acgt"],
                "ambiguous_fraction": row["ambiguous_fraction"],
                "exon_count": row["exon_count"], "intron_count": row["exon_count"] - 1,
                "transcript_count": tx_count[gid], **kmer_frequencies(row["sequence"])})
    return representatives

representatives = export_all(genes, transcripts, exons, genome, OUTPUT)
print("Exported genomic genes:", len(genes))
print("Exported cDNA transcripts:", len(transcripts))
print("Feature-table genes:", len(representatives))
for path in sorted(OUTPUT.iterdir()):
    print(path.name, round(path.stat().st_size / 1e6, 2), "MB")


## 6. Inspect and export MOC1
This checks the MOC1 model in **your MSU Release 7 annotation**. A gene-model sequence can differ from a curated canonical sequence. Verify coding structure and curated MONOCULM1 identity before functional or protein analyses. MOC1's presence in a reference file does not indicate its expression in your stem sample.

In [ ]:
MOC1_ID = "LOC_Os06g40780"
if MOC1_ID not in genes:
    raise ValueError("MOC1 ID not present: verify annotation/ID system.")
rec = genes[MOC1_ID]
sequence = extract_interval(genome, rec["chromosome"], rec["start"], rec["end"], rec["strand"])
print("MOC1 annotation:", rec)
print("Genomic length:", len(sequence))
with open(OUTPUT / "MOC1_genomic.fasta", "w") as f:
    write_fasta(f, MOC1_ID, sequence, "annotated_genomic_locus")
with open(OUTPUT / "MOC1_transcripts.fasta", "w") as f:
    for tid, tx in transcripts.items():
        if tx["gene_id"] == MOC1_ID:
            sequence = transcript_sequence(tid, transcripts, exons, genome)
            write_fasta(f, tid, sequence, f"gene_id={MOC1_ID} spliced_cdna")
            print("Transcript:", tid, "length:", len(sequence), "exons:", len(set(exons[tid])))
            print("First 80 bases:", sequence[:80])


## 7. Record provenance and download results
The manifest records input hashes, counts and extraction rules. The ZIP may be large. Keep the original reference inputs together with the outputs for reproducibility.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = {"annotation_file": gff_path.name, "annotation_sha256": sha256_file(gff_path),
    "genome_file": genome_path.name, "genome_sha256": sha256_file(genome_path),
    "gene_count": len(genes), "transcript_count": len(transcripts),
    "feature_counts": dict(feature_counts),
    "coordinates": "GFF3 1-based inclusive; outputs oriented on gene/transcript strand",
    "genomic_scope": "annotated gene boundaries, including introns and annotated UTRs; no added flanks",
    "transcript_scope": "spliced annotated exons; UTRs included",
    "ml_representative": "longest spliced cDNA per gene; ties by transcript ID",
    "kmer_rule": "overlapping 3-mers; exclude windows with non-ACGT characters; normalize by valid windows",
    "expression_measured": False, "supervised_model_trained": False,
    "reference_source": "https://rice.uga.edu/download_osa1r7.shtml"}
(OUTPUT / "provenance.json").write_text(json.dumps(manifest, indent=2))
zip_path = shutil.make_archive(str(ROOT / "rice_gene_sequences_and_features"), "zip", OUTPUT)
print("Download:", zip_path)
files.download(zip_path)


## Optional: save results in your own Google Drive
Run the following cell only if you want a durable copy in Drive. Colab will ask you to authorize Drive access. The browser download above does not require Drive access.

In [ ]:
# Uncomment these lines if you want to save the ZIP to Google Drive.
# from google.colab import drive
# drive.mount("/content/drive")
# destination = Path("/content/drive/MyDrive/Rice_MOC1_project")
# destination.mkdir(parents=True, exist_ok=True)
# shutil.copy2(zip_path, destination / Path(zip_path).name)
# print("Saved:", destination / Path(zip_path).name)


## 8. How to use the features in a later ML study

Choose a concrete prediction question before model training. The GFF3 supplies IDs and structural annotations; it does **not** supply validated lodging labels or rice-variety phenotypes.

- **Gene-level candidate prioritization:** combine these sequence/structure features with stem expression, domains, pathways and evidence-backed labels. Unstudied genes are unlabeled, not automatically negative. MOC1 as the sole positive example is insufficient to train a defensible classifier.
- **Variety-level lodging prediction:** each observation must be a rice variety/accession with measured lodging or stem-strength phenotype. Use matched genotype/haplotype/expression features across accessions; one Nipponbare reference cannot provide between-variety variation.
- **Exploration without labels:** clustering or PCA can describe patterns but does not establish a lodging function.

Keep gene IDs as join keys, not biological numeric predictors. Drop annotation descriptions or any features containing the answer from model predictors. Keep all isoforms of the same gene together during splitting; closely related gene families/homology clusters should be grouped when evaluating generalization to new families. Fit normalization, imputation and feature selection on training data only. A held-out test set is required for a predictive claim.

References:
- Rice files and assembly: https://rice.uga.edu/download_osa1r7.shtml
- Rice expression atlas: https://rice.uga.edu/expression.shtml
- Grouped evaluation: https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data

No classifier is trained here because no validated target labels have been supplied.
